# PRiSM Track B Closed-Loop (Thin Colab Notebook)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/thesis-research-colab/blob/main/notebooks/PRiSM_trackB_closedloop_simulation_colab.ipynb)

This notebook is intentionally thin. Core logic lives in `src/trackb/*.py`.


## Notebook Purpose And Contract

This notebook is intentionally orchestration-only and keeps heavy logic in reusable Python modules.

Execution contract:
1. Sync latest code into `/content/thesis-research-colab`.
2. Mount Google Drive so resume artifacts survive runtime resets.
3. Bootstrap a deterministic environment only when needed.
4. Run calibration and simulation with stable run identifiers.
5. Export artifacts required for analysis and paper figures.


## Step 1 - Sync Repository

This cell keeps the runtime aligned with the current `main` branch.

Behavior:
- If repo directory already exists, it does `fetch + checkout main + pull --ff-only`.
- Otherwise it clones a fresh copy.

If this repo is private, you must authenticate git in Colab before this step.


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/achyutmorang/thesis-research-colab.git"
REPO_DIR = "/content/thesis-research-colab"

if os.path.exists(REPO_DIR):
    print("Repo exists, fast-forwarding main...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


## Step 2 - Mount Persistent Storage

Artifacts such as checkpoints, per-scenario outputs, traces, and thresholds are written under `PERSIST_ROOT`.

Why this matters:
- Colab VM storage (`/content`) is ephemeral.
- Drive-mounted paths (`/content/drive/...`) let you resume runs after interruptions.


In [ ]:
# Auto-mount Google Drive for persistent resume artifacts (Colab only)
import os

if os.environ.get('COLAB_RELEASE_TAG'):
    try:
        from google.colab import drive
        if not os.path.ismount('/content/drive'):
            drive.mount('/content/drive')
        else:
            print('[drive] /content/drive already mounted')
    except Exception as e:
        print('[drive] mount skipped:', e)
else:
    print('[drive] non-Colab environment; skipping mount')


## Step 3 - Deterministic Runtime Setup

Setup is offloaded to `scripts/colab_setup.py` to keep this notebook readable and reviewable.

What setup does when `RUN_SETUP=True`:
- installs pinned Waymax/JAX/LatentDriver-compatible dependencies,
- patches LatentDriver compatibility points,
- fetches the expected checkpoint if missing.

Run this only in a fresh runtime, then restart once and set `RUN_SETUP=False`.


In [ ]:
# Deterministic setup (run ONCE in a fresh Colab runtime)
# Set RUN_SETUP=True for the first run in a new runtime, then restart once.
from pathlib import Path
import importlib.util

RUN_SETUP = False

setup_py = Path.cwd() / "scripts" / "colab_setup.py"
if not setup_py.exists():
    raise FileNotFoundError(f"Setup helper missing: {setup_py}")

spec = importlib.util.spec_from_file_location("colab_setup", str(setup_py))
if spec is None or spec.loader is None:
    raise RuntimeError(f"Unable to load setup helper from {setup_py}")

colab_setup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_setup)
colab_setup.run_deterministic_setup(run_setup=RUN_SETUP)


### Optional Manual Checkpoint Fallback

Use this only if automatic checkpoint download fails.


In [ ]:
# Manual checkpoint fallback (run only if setup could not fetch checkpoint)
# !mkdir -p /content/checkpoints
# !wget -O /content/checkpoints/lantentdriver_t2_J3.ckpt https://huggingface.co/Sephirex-x/LatentDriver/resolve/main/checkpoints/lantentdriver_t2_J3.ckpt
# !ls -lh /content/checkpoints/lantentdriver_t2_J3.ckpt


## Step 4 - Import Modular Pipeline APIs

All experiment logic is imported from `src/trackb` modules:
- configuration,
- calibration,
- search,
- export/resume utilities.


In [ ]:
import numpy as np
import pandas as pd

from src.trackb import (
    SearchConfig,
    build_trackb_runner_and_splits,
    auto_select_shard_id,
    configure_persistent_run_prefix,
    inspect_shard_progress,
    export_trackb_artifacts,
    initialize_configs,
    make_waymax_data_iter,
    run_preflight_and_calibration,
    run_surprise_quality_gate,
    run_trackb_closed_loop,
    summarize_method_outputs,
)


## Step 5 - Configure Run Identity And Persistence

Key controls:
- `RUN_TAG`: experiment identifier used in artifact filenames.
- `PERSIST_ROOT`: persistent artifact root (typically Google Drive).
- `N_SHARDS`: how many disjoint shard runs to split evaluation into.
- `SHARD_ID`: use `"auto"` to resume the next shard automatically, or set an explicit integer.
- `RESTORE_FROM_UPLOAD`: manual fallback for restoring missing artifacts.

Auto shard selection behavior:
- If any shard has no result file yet, it picks the first missing shard.
- Otherwise, it picks the least-complete shard based on existing result files.


In [ ]:
cfg, search_cfg, ckpt_scan_df = initialize_configs()

RUN_TAG = "trackb_closedloop_v1"
PERSIST_ROOT = "/content/drive/MyDrive/prism_trackb_runs/v1"
N_SHARDS = 5
SHARD_ID = "auto"   # set int to force a specific shard; use "auto" to resume next shard automatically
RESTORE_FROM_UPLOAD = False

shard_progress_df = inspect_shard_progress(
    run_tag=RUN_TAG,
    persist_root=PERSIST_ROOT,
    n_shards=N_SHARDS,
)

if isinstance(SHARD_ID, str) and SHARD_ID.strip().lower() == "auto":
    SHARD_ID = auto_select_shard_id(
        run_tag=RUN_TAG,
        persist_root=PERSIST_ROOT,
        n_shards=N_SHARDS,
    )
    print(f"[shard] auto-selected SHARD_ID={SHARD_ID}")
else:
    SHARD_ID = int(SHARD_ID)

run_prefix = configure_persistent_run_prefix(
    cfg=cfg,
    run_tag=RUN_TAG,
    persist_root=PERSIST_ROOT,
    shard_id=SHARD_ID,
    n_shards=N_SHARDS,
)
print("run_prefix =", run_prefix)
print(f"[shard] running {SHARD_ID + 1}/{max(1, N_SHARDS)}")

if len(shard_progress_df):
    display(shard_progress_df)

if len(ckpt_scan_df):
    display(ckpt_scan_df.head(10))


## Step 6 - Build Dataset, Splits, And Scenario Handles

This stage:
- validates WOMD GCS access,
- builds train/eval splits,
- retains simulator states for eval scenarios,
- prepares reference/open-loop baselines.


In [ ]:
dataset_config, data_iter = make_waymax_data_iter(cfg)
(
    runner,
    data,
    train_idx,
    test_idx,
    eval_idx_all,
    eval_idx,
    reference_df,
    base_eval_openloop_df,
) = build_trackb_runner_and_splits(
    cfg=cfg,
    data_iter=data_iter,
    dataset_config=dataset_config,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
)

print(f"train={len(train_idx)} test={len(test_idx)} eval_shard={len(eval_idx)}")


## Step 7 - Preflight Checks And Closed-Loop Calibration

Preflight verifies planner and rollout health before expensive simulation.
Calibration estimates risk/surprise thresholds and scaling used by search objectives.


In [ ]:
(
    preflight_df,
    closedloop_calib_df,
    trackb_thresholds,
    calib_diag_df,
    calib_quant_df,
) = run_preflight_and_calibration(
    runner=runner,
    cfg=cfg,
    search_cfg=search_cfg,
    eval_idx=eval_idx,
    reference_df=reference_df,
    restore_from_upload=RESTORE_FROM_UPLOAD,
)

print("Calibration thresholds:", trackb_thresholds)
display(preflight_df)
display(calib_diag_df)
display(calib_quant_df)


## Step 8 - Surprise Quality Gate

This gate prevents running full optimization when surprise signal quality is degenerate
(for example near-zero variance or excessive fallback behavior).


In [ ]:
gate_summary_df, dist_change_summary_df = run_surprise_quality_gate(
    closedloop_calib_df=closedloop_calib_df,
    surprise_gate_enabled=True,
)
display(gate_summary_df)
display(dist_change_summary_df)


## Step 9 - Closed-Loop Optimization Run

Runs method comparisons under matched budgets and common-random-number controls,
with periodic checkpointing for resume robustness.


In [ ]:
trackb_results_df, trackb_trace_df = run_trackb_closed_loop(
    runner=runner,
    eval_idx=eval_idx,
    cfg=cfg,
    search_cfg=search_cfg,
    thresholds=trackb_thresholds,
    run_prefix=cfg.run_prefix,
    static_frames={
        "base_eval_openloop_df": base_eval_openloop_df,
        "reference_df": reference_df,
        "closedloop_calib_df": closedloop_calib_df,
        "preflight_df": preflight_df,
        "calib_diag_df": calib_diag_df,
        "calib_quant_df": calib_quant_df,
    },
)

print("Result rows:", len(trackb_results_df))
print("Trace rows:", len(trackb_trace_df))


## Step 10 - Summarize And Export Artifacts

Exports include per-scenario outputs, traces, thresholds, diagnostics, and runtime manifest.
These files are the primary inputs for paper tables/plots and reproducibility records.


In [ ]:
quick_summary_df, sanity_df, fairness_checks_df, trace_diag_df = summarize_method_outputs(
    trackb_results_df=trackb_results_df,
    trackb_trace_df=trackb_trace_df,
)

artifact_paths = export_trackb_artifacts(
    cfg=cfg,
    search_cfg=search_cfg,
    eval_idx=eval_idx,
    trackb_results_df=trackb_results_df,
    trackb_trace_df=trackb_trace_df,
    base_eval_openloop_df=base_eval_openloop_df,
    reference_df=reference_df,
    closedloop_calib_df=closedloop_calib_df,
    preflight_df=preflight_df,
    calib_diag_df=calib_diag_df,
    calib_quant_df=calib_quant_df,
    trackb_thresholds=trackb_thresholds,
    quick_summary_df=quick_summary_df,
    sanity_df=sanity_df,
    fairness_checks_df=fairness_checks_df,
    trace_diag_df=trace_diag_df,
)

display(quick_summary_df)
display(sanity_df)
display(fairness_checks_df)
display(trace_diag_df)

print("Artifacts:")
for key, value in artifact_paths.items():
    print(f" - {key}: {value}")
